# Análisis de Caso – Big Data (MarketTrend S.A.)
### Arquitectura Big Data en la nube (data lake en S3 como ejemplo)

Este notebook desarrolla paso a paso el análisis del caso de Big Data para la empresa MarketTrend S.A., usando una arquitectura **cloud** basada en data lake (por ejemplo, Amazon S3), motores de procesamiento distribuido (Apache Spark) y un bus de eventos (Apache Kafka).

## 1. Análisis de la situación y las 5V del Big Data

En esta sección describimos las 5V presentes en el caso: Volumen, Velocidad, Variedad, Veracidad y Valor. Cada punto incluye una breve interpretación y un mini–ejemplo numérico.

### 1.1 Volumen
1. MarketTrend S.A. recibe grandes cantidades de datos desde redes sociales, registros de compras, aplicaciones móviles y encuestas online.
2. Esta cantidad supera la capacidad de los sistemas tradicionales para almacenar y procesar la información de forma eficiente.
3. **Interpretación:** Volumen se refiere a la cantidad total de datos que manejamos.
4. **Mini–ejemplo:** si cada día se generan 2 millones de eventos y la base de datos solo procesa 50 mil, tenemos un problema clásico de volumen.

### 1.2 Velocidad
1. Los datos llegan de forma continua desde redes sociales y apps móviles.
2. La empresa necesita detectar tendencias en tiempo casi real, no solo en análisis históricos.
3. **Interpretación:** Velocidad es el ritmo al que los datos se generan, transmiten y deben procesarse.
4. **Mini–ejemplo:** si cada segundo entran 500 nuevos eventos, no basta con procesarlos una vez al día; se necesitan procesos de streaming.

### 1.3 Variedad
1. Las fuentes de datos incluyen: textos de redes sociales, registros estructurados de compras, logs y eventos de apps, y respuestas de encuestas.
2. Esto implica mezclar datos estructurados, semiestructurados y no estructurados en una misma solución.
3. **Interpretación:** Variedad alude a los distintos tipos y formatos de datos.
4. **Mini–ejemplo:** combinar un CSV de ventas, un JSON de eventos móviles y comentarios de texto libre en una analítica integrada.

### 1.4 Veracidad
1. Los datos de redes sociales y encuestas pueden contener ruido, errores, bots o respuestas poco sinceras.
2. Si la calidad de datos es baja, las recomendaciones y análisis serán poco confiables.
3. **Interpretación:** Veracidad mide la confiabilidad y calidad de los datos.
4. **Mini–ejemplo:** si el 20% de las compras tiene el cliente mal identificado, los modelos de segmentación fallarán.

### 1.5 Valor
1. El objetivo del proyecto es mejorar la toma de decisiones estratégicas y ofrecer recomendaciones personalizadas a los clientes.
2. El dato, por sí solo, no es suficiente: debe convertirse en insights accionables.
3. **Interpretación:** Valor es el beneficio real que obtenemos al usar los datos (más ventas, mejor experiencia, menos costos).
4. **Mini–ejemplo:** si, gracias al análisis, aumentamos en 10% la tasa de conversión de campañas, los datos están aportando valor concreto.

## 2. Arquitectura Big Data en la nube

A continuación definimos una arquitectura por capas pensada para ejecutarse en la nube. Tomaremos como referencia un data lake en Amazon S3 (el concepto aplica también a otros proveedores).

Las capas serán:
1. Capa de adquisición / ingesta de datos.
2. Capa de almacenamiento distribuido (data lake en S3).
3. Capa de procesamiento (batch y streaming) con Spark y/o Flink.
4. Capa de analítica y consumo (BI y modelos ML).

## 3. Capa de adquisición / ingesta de datos

Primero distinguimos dos grandes modos de ingesta: batch (lotes periódicos) y streaming (flujo continuo).

### 3.1 Ingesta en streaming (eventos en tiempo real)
1. Redes sociales y apps móviles generan eventos continuamente (clicks, likes, vistas, etc.).
2. Para capturar estos eventos usamos un bus de mensajes como Apache Kafka.
3. Los productores (aplicaciones backend, conectores de redes sociales) envían eventos a tópicos de Kafka.
4. Los consumidores leen desde Kafka y escriben los datos en el data lake en S3 para su posterior análisis.

### 3.2 Ingesta batch (lotes periódicos)
1. Los registros de compras y encuestas pueden exportarse en archivos CSV/Parquet de forma diaria o periódica.
2. Usamos procesos ETL o herramientas específicas (por ejemplo, AWS Glue, jobs programados) para mover estos archivos hacia S3.
3. Una vez en S3, estos datos se organizan en carpetas por fecha y tipo de fuente, facilitando su lectura posterior por Spark u otras herramientas.

In [1]:
# Ejemplo ilustrativo (no ejecutable) de definición de fuentes de datos en Python
# La idea es mostrar cómo podríamos representar las fuentes de ingesta de forma estructurada.

fuentes_streaming = {
    'redes_sociales': {
        'tipo': 'streaming',
        'tecnologia_ingesta': 'Kafka',
        'topico_kafka': 'eventos_redes_sociales'
    },
    'app_movil': {
        'tipo': 'streaming',
        'tecnologia_ingesta': 'Kafka',
        'topico_kafka': 'eventos_app_movil'
    }
}

fuentes_batch = {
    'registros_compras': {
        'tipo': 'batch',
        'formato_archivo': 'parquet',
        'frecuencia': 'diaria'
    },
    'encuestas_online': {
        'tipo': 'batch',
        'formato_archivo': 'csv',
        'frecuencia': 'semanal'
    }
}

print('Fuentes en streaming:', fuentes_streaming)
print('Fuentes batch:', fuentes_batch)

Fuentes en streaming: {'redes_sociales': {'tipo': 'streaming', 'tecnologia_ingesta': 'Kafka', 'topico_kafka': 'eventos_redes_sociales'}, 'app_movil': {'tipo': 'streaming', 'tecnologia_ingesta': 'Kafka', 'topico_kafka': 'eventos_app_movil'}}
Fuentes batch: {'registros_compras': {'tipo': 'batch', 'formato_archivo': 'parquet', 'frecuencia': 'diaria'}, 'encuestas_online': {'tipo': 'batch', 'formato_archivo': 'csv', 'frecuencia': 'semanal'}}


## 4. Capa de almacenamiento: Data Lake en S3

En la nube, un patrón muy común es centralizar todos los datos en un data lake sobre almacenamiento de objetos (por ejemplo, Amazon S3).

### 4.1 Organización por zonas (raw, clean, curated)
1. Zona **raw**: almacena los datos tal como llegan de las fuentes, sin transformaciones.
2. Zona **clean/silver**: almacena datos limpios y estandarizados (tipos corregidos, duplicados eliminados).
3. Zona **curated/gold**: almacena datasets listos para analítica avanzada o modelos ML (tablas de clientes, segmentos, features).
4. Esta división ayuda a mantener trazabilidad y a separar claramente los distintos niveles de calidad de los datos.

In [2]:
# Ejemplo ilustrativo de una estructura de rutas en un data lake en S3

data_lake_s3 = {
    'raw': {
        'redes_sociales': 's3://markettrend-datalake/raw/redes_sociales/',
        'app_movil': 's3://markettrend-datalake/raw/app_movil/',
        'registros_compras': 's3://markettrend-datalake/raw/registros_compras/',
        'encuestas': 's3://markettrend-datalake/raw/encuestas/'
    },
    'clean': {
        'interacciones': 's3://markettrend-datalake/clean/interacciones/',
        'compras': 's3://markettrend-datalake/clean/compras/',
        'clientes': 's3://markettrend-datalake/clean/clientes/'
    },
    'curated': {
        'segmentos_clientes': 's3://markettrend-datalake/curated/segmentos_clientes/',
        'features_ml': 's3://markettrend-datalake/curated/features_ml/'
    }
}

for zona, rutas in data_lake_s3.items():
    print(f'Zona {zona}:')
    for nombre, ruta in rutas.items():
        print(f'  - {nombre}: {ruta}')

Zona raw:
  - redes_sociales: s3://markettrend-datalake/raw/redes_sociales/
  - app_movil: s3://markettrend-datalake/raw/app_movil/
  - registros_compras: s3://markettrend-datalake/raw/registros_compras/
  - encuestas: s3://markettrend-datalake/raw/encuestas/
Zona clean:
  - interacciones: s3://markettrend-datalake/clean/interacciones/
  - compras: s3://markettrend-datalake/clean/compras/
  - clientes: s3://markettrend-datalake/clean/clientes/
Zona curated:
  - segmentos_clientes: s3://markettrend-datalake/curated/segmentos_clientes/
  - features_ml: s3://markettrend-datalake/curated/features_ml/


## 5. Capa de procesamiento distribuido (Spark en la nube)

Usaremos Apache Spark como motor principal de procesamiento batch y, opcionalmente, Spark Structured Streaming para la parte de tiempo real.

### 5.1 Procesamiento batch
1. Spark lee datos desde la zona raw/clean del data lake en S3.
2. Aplica transformaciones: limpieza, agregaciones, cálculo de métricas por cliente.
3. Escribe los resultados en la zona curated/gold.
4. Estos resultados se usan para dashboards y modelos ML.

In [3]:
# Ejemplo simplificado en PySpark (pseudocódigo, no se ejecuta aquí)

spark_code_batch = '''
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('MarketTrendBatch').getOrCreate()

# 1. Leer datos de compras desde S3 (zona clean)
df_compras = spark.read.parquet('s3://markettrend-datalake/clean/compras/')

# 2. Calcular métricas por cliente (ejemplo RFM simplificado)
from pyspark.sql import functions as F

df_metricas = df_compras.groupBy('id_cliente').agg(
    F.count('*').alias('frecuencia_compras'),
    F.max('fecha_compra').alias('ultima_compra'),
    F.avg('monto').alias('ticket_promedio')
)

# 3. Guardar resultados en la zona curated
df_metricas.write.mode('overwrite').parquet('s3://markettrend-datalake/curated/metricas_clientes/')

spark.stop()
'''

print(spark_code_batch)


from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('MarketTrendBatch').getOrCreate()

# 1. Leer datos de compras desde S3 (zona clean)
df_compras = spark.read.parquet('s3://markettrend-datalake/clean/compras/')

# 2. Calcular métricas por cliente (ejemplo RFM simplificado)
from pyspark.sql import functions as F

df_metricas = df_compras.groupBy('id_cliente').agg(
    F.count('*').alias('frecuencia_compras'),
    F.max('fecha_compra').alias('ultima_compra'),
    F.avg('monto').alias('ticket_promedio')
)

# 3. Guardar resultados en la zona curated
df_metricas.write.mode('overwrite').parquet('s3://markettrend-datalake/curated/metricas_clientes/')

spark.stop()



### 5.2 Procesamiento streaming (Spark Structured Streaming)
1. Spark lee en tiempo real desde Kafka los eventos de interacción (por ejemplo, de la app móvil).
2. Define ventanas de tiempo para calcular métricas en streaming (productos más vistos en los últimos 5 minutos, por ejemplo).
3. Escribe los resultados en tablas que pueden ser consultadas por servicios de recomendación.

In [4]:
# Ejemplo simplificado de Spark Structured Streaming leyendo desde Kafka (pseudocódigo)

spark_code_streaming = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark = SparkSession.builder.appName('MarketTrendStreaming').getOrCreate()

# 1. Leer eventos desde Kafka
df_raw = spark.readStream.format('kafka')\
    .option('kafka.bootstrap.servers', 'kafka:9092')\
    .option('subscribe', 'eventos_app_movil')\
    .load()

# 2. Definir esquema del valor
schema = StructType([
    StructField('id_cliente', StringType()),
    StructField('producto', StringType()),
    StructField('timestamp', TimestampType())
])

df_parsed = df_raw.select(
    from_json(col('value').cast('string'), schema).alias('data')
).select('data.*')

# 3. Calcular productos más vistos en ventanas de 5 minutos
df_agg = df_parsed.groupBy(
    window(col('timestamp'), '5 minutes'),
    col('producto')
).count().orderBy(col('count').desc())

# 4. Escribir resultados en consola o en un sink (por ejemplo, tabla en S3 o base NoSQL)
query = df_agg.writeStream.outputMode('complete')\
    .format('console')\
    .start()

query.awaitTermination()
'''

print(spark_code_streaming)


from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark = SparkSession.builder.appName('MarketTrendStreaming').getOrCreate()

# 1. Leer eventos desde Kafka
df_raw = spark.readStream.format('kafka')    .option('kafka.bootstrap.servers', 'kafka:9092')    .option('subscribe', 'eventos_app_movil')    .load()

# 2. Definir esquema del valor
schema = StructType([
    StructField('id_cliente', StringType()),
    StructField('producto', StringType()),
    StructField('timestamp', TimestampType())
])

df_parsed = df_raw.select(
    from_json(col('value').cast('string'), schema).alias('data')
).select('data.*')

# 3. Calcular productos más vistos en ventanas de 5 minutos
df_agg = df_parsed.groupBy(
    window(col('timestamp'), '5 minutes'),
    col('producto')
).count().orderBy(col('count').desc())

# 4. Escribir resultados en consola o en un sink (por ejemplo, tabla en 

## 6. Capa de analítica y consumo

Sobre los datos de la zona curated/gold del data lake podemos construir dashboards de BI y modelos de Machine Learning.

### 6.1 Dashboards de BI
1. Herramientas como Power BI, Tableau o Looker se conectan al data lake o a un data warehouse derivado.
2. Permiten visualizar tendencias de clientes, ventas por segmento, rendimiento de campañas, etc.
3. Estos dashboards apoyan la toma de decisiones estratégicas.

### 6.2 Modelos de Machine Learning
1. Con los datos integrados (compras, interacciones, encuestas) se pueden entrenar modelos de segmentación, recomendación y churn.
2. Los modelos pueden entrenarse en Spark MLlib o en otras plataformas de ML que lean desde el data lake.
3. Los resultados (por ejemplo, segmentos de clientes) se devuelven a la zona curated para ser consumidos por otros sistemas.

## 7. Beneficios, riesgos y medidas (resumen para el caso)

A modo de cierre, recordamos los beneficios, riesgos y medidas asociados a esta arquitectura Big Data en la nube.

### 7.1 Beneficios
1. Análisis casi en tiempo real gracias a Kafka + Spark Streaming.
2. Escalabilidad horizontal del data lake en S3, permitiendo crecer en volumen y variedad de datos.
3. Mejor calidad de recomendaciones y decisiones estratégicas al integrar múltiples fuentes de datos.

### 7.2 Riesgos
1. Seguridad y privacidad de los datos (posibles brechas y uso indebido).
2. Calidad de datos (ruido, inconsistencias, duplicados).
3. Complejidad tecnológica y necesidad de perfiles especializados.

### 7.3 Medidas recomendadas
1. Implementar control de acceso basado en roles (RBAC) y cifrado en tránsito y en reposo.
2. Definir pipelines de limpieza y validación de datos, más un catálogo de datos para trazabilidad.
3. Establecer políticas de gobernanza de datos y programas de formación en privacidad y seguridad.